# 15. Blessed 130M ladder runs (baseline / Full / Block × 2 seeds)

Goal: run the **130M Moonshot-shape ladder config** for all three architectures at **~808M tokens each** (~5 tokens/param), with **inline validation loss**, **per-layer magnitude/gradient logging**, **throughput logging**, and **NEW layer-1 depth-attention source logging** (to track whether the 90M Full `‖h₁‖` spike persists). Real results run — **no downstream benchmark eval during training**.

- **Model**: ~161.7M params (`d_model=896`, `n_layers=12`, `n_heads=14`, `d_ff=3584`)
- **Hyperparameters**: `configs/fineweb_130m_ladder.yaml` (AdamW 0.9/0.95, wd 0.1, eps 1e-8, cosine + warmup, bf16, effective batch 65,536 tokens)
- **Variants × seeds**: `baseline`, `attnres` (Full), `block_attnres` (Block, `num_blocks=8` → balanced `[2,2,2,2,1,1,1,1]`) × seeds `{123, 1337}` = **6 runs**
- **Block layout (FIXED)**: `_block_sizes(12, 8) == [2,2,2,2,1,1,1,1]` — **not** the old broken `[1,1,1,1,1,1,1,5]`
- **Dataset**: `fineweb_edu` sample-10BT, `block_size=1024`, existing hash split
- **Logging**: train loss every 20 steps, **val loss every 100 steps**, per-layer `h_l` + `dL/dh_l`, `tokens_per_sec` + wall-clock, **layer-1 depth-attn weights + source magnitudes**, checkpoints + summaries to **Google Drive**

Flow: sync repo → mount Drive → GPU check → pre-launch report (config / block layout / 6-run plan / cost) → VRAM probe → **confirmation gate** → launch sequentially.

> Notebook 14 = 90M off-curve. This notebook is the next ladder point (130M).


In [ ]:
import os
import sys
import time
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/AtinChing/AttnResGPT-next-scale.git"
REPO_TARGET = Path("/content/AttnResGPT-next-scale")
DRIVE_ARTIFACT_FOLDER = "AttnResGPT-next-scale-artifacts"


def is_repo_root(path: Path) -> bool:
    return (path / "src" / "training" / "train.py").exists() and (path / "requirements.txt").exists()


def discover_repo() -> Path:
    for candidate in [REPO_TARGET, Path.cwd(), Path.cwd().parent]:
        if is_repo_root(candidate):
            return candidate.resolve()
    if not REPO_TARGET.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_TARGET)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_TARGET), "pull", "--ff-only"], check=False)
    return REPO_TARGET.resolve()


REPO = discover_repo()
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# W&B: prefer Colab secret / env. Do not hardcode keys in the notebook.
try:
    from google.colab import userdata
    if not os.environ.get("WANDB_API_KEY"):
        os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
except Exception:
    pass

print("repo:", REPO)
print("WANDB_API_KEY set =", bool(os.environ.get("WANDB_API_KEY")))


In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive") / DRIVE_ARTIFACT_FOLDER
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("drive root:", DRIVE_ROOT)
print("artifacts layout:")
print("  checkpoints/", DRIVE_ROOT / "checkpoints")
print("  runs/       ", DRIVE_ROOT / "runs")
print("  logs/       ", DRIVE_ROOT / "logs")


In [ ]:
import torch

print("cuda_available =", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("device_name =", props.name)
    print("total_vram_gib =", round(props.total_memory / (1024 ** 3), 1))
    print("bf16_supported =", torch.cuda.is_bf16_supported())


## Pre-launch report: 130M config + balanced block layout + 6-run plan + logging + cost

Review this cell's output before enabling the confirmation gate. **Do not launch until you confirm.**


In [ ]:
from src.utils.config import load_config
from src.models.attnres import build_model, _block_sizes
from src.utils.logging import build_run_identity
from src.utils.runtime import count_parameters

CONFIG_PATH = "configs/fineweb_130m_ladder.yaml"
EFFECTIVE_BATCH_TOKENS = 65_536
SEEDS = [123, 1337]

RUN_PLAN = [
    {
        "label": "baseline",
        "architecture": "baseline",
        "overrides": ["model.architecture=baseline", "model.attnres.enabled=false"],
        "notes": "Standard PreNorm residual GPT.",
    },
    {
        "label": "full_attnres",
        "architecture": "attnres",
        "overrides": [
            "model.architecture=attnres",
            "model.attnres.enabled=true",
            "model.attnres.mode=full",
        ],
        "notes": "Full depth-attention over all prior sublayer outputs.",
    },
    {
        "label": "block_attnres",
        "architecture": "block_attnres",
        "overrides": ["model.architecture=block_attnres"],
        "notes": "Block AttnRes (num_blocks=8 → balanced [2,2,2,2,1,1,1,1]).",
    },
]

cfg = load_config(CONFIG_PATH, overrides=[f"experiment.seed={SEEDS[0]}"])
eff_tokens = cfg.data.batch_size * cfg.training.grad_accum_steps * cfg.data.block_size
assert eff_tokens == EFFECTIVE_BATCH_TOKENS, f"effective batch is {eff_tokens}, expected {EFFECTIVE_BATCH_TOKENS}"
assert cfg.training.eval_interval > 0, "eval_interval must be > 0 for inline val loss"
assert cfg.evaluation.positionwise_steps == [], "no positionwise / benchmark steps during training"
assert cfg.model.attnres.num_blocks == 8, "Block AttnRes expects num_blocks=8 in YAML"
assert cfg.model.n_layers == 12 and cfg.model.d_model == 896

EXPECTED_BLOCK_SIZES = [2, 2, 2, 2, 1, 1, 1, 1]
BROKEN_BLOCK_SIZES = [1, 1, 1, 1, 1, 1, 1, 5]
actual_block_sizes = _block_sizes(cfg.model.n_layers, cfg.model.attnres.num_blocks)
assert actual_block_sizes == EXPECTED_BLOCK_SIZES, actual_block_sizes
assert actual_block_sizes != BROKEN_BLOCK_SIZES

param_counts = {}
block_sizes_by_variant = {}
for plan in RUN_PLAN:
    variant_cfg = load_config(CONFIG_PATH, overrides=[f"experiment.seed={SEEDS[0]}", *plan["overrides"]])
    model = build_model(variant_cfg.model)
    param_counts[plan["label"]] = count_parameters(model)["total"]
    block_sizes_by_variant[plan["label"]] = getattr(model, "block_sizes", None)

run_names = {}
for seed in SEEDS:
    for plan in RUN_PLAN:
        variant_cfg = load_config(CONFIG_PATH, overrides=[f"experiment.seed={seed}", *plan["overrides"]])
        run_names[(seed, plan["label"])] = build_run_identity(variant_cfg).run_name

tokens_per_run = cfg.training.max_steps * EFFECTIVE_BATCH_TOKENS
tokens_total = len(SEEDS) * len(RUN_PLAN) * tokens_per_run

# Wall-clock from *measured* 90M A100 elapsed_time_sec (mean of seeds 123/1337),
# scaled by (tokens × params). Same L=12 depth-attn geometry as 90M Full/Block.
WALL_90M_HOURS = {
    "baseline": ((8261.5 + 8380.5) / 2) / 3600.0,       # ~2.31 h
    "full_attnres": ((29237.3 + 29234.2) / 2) / 3600.0, # ~8.12 h
    "block_attnres": ((15321.8 + 15346.9) / 2) / 3600.0,# ~4.26 h
}
PARAMS_90M = 91_887_360
TOKENS_90M = 459_407_360
token_scale = tokens_per_run / TOKENS_90M
param_scale = param_counts["baseline"] / PARAMS_90M
wall_scale = token_scale * param_scale
est_hours = {label: hours * wall_scale for label, hours in WALL_90M_HOURS.items()}
suite_hours_est = sum(est_hours[p["label"]] for p in RUN_PLAN for _ in SEEDS)
A100_USD_PER_HOUR = 2.0

print("=== 130M ladder config ===")
print(f"config file     : {CONFIG_PATH}")
print(f"model shape     : d_model={cfg.model.d_model} n_layers={cfg.model.n_layers} "
      f"n_heads={cfg.model.n_heads} d_ff={cfg.model.d_ff}")
print(f"param counts    : baseline={param_counts['baseline']:,}  "
      f"full={param_counts['full_attnres']:,}  block={param_counts['block_attnres']:,}")
print(f"num_blocks YAML : {cfg.model.attnres.num_blocks}")
print(f"block layout    : {actual_block_sizes}  ✓ balanced (NOT {BROKEN_BLOCK_SIZES})")
print(f"  build_model Block.block_sizes = {block_sizes_by_variant['block_attnres']}")
print(f"dataset         : {cfg.data.dataset_name} / {cfg.data.fineweb_subset}  "
      f"block_size={cfg.data.block_size}  hash_modulo={cfg.data.hash_modulo}  "
      f"val_fraction={cfg.data.val_fraction}")
print(f"optimizer       : AdamW lr={cfg.training.learning_rate} min_lr={cfg.training.min_lr} "
      f"betas=({cfg.training.beta1},{cfg.training.beta2}) wd={cfg.training.weight_decay} "
      f"eps={cfg.training.adam_eps} clip={cfg.training.grad_clip}")
print(f"schedule        : cosine  warmup={cfg.training.warmup_steps}  "
      f"max_steps={cfg.training.max_steps}  amp={cfg.training.amp_dtype}")
print(f"YAML microbatch : {cfg.data.batch_size} x {cfg.training.grad_accum_steps} accum "
      f"x {cfg.data.block_size} ctx = {eff_tokens:,} tokens/step")
print(f"horizon/run     : {tokens_per_run:,} tokens ({tokens_per_run/1e6:.1f}M)  "
      f"~{tokens_per_run / param_counts['baseline']:.2f} tokens/param")
print(f"total budget    : {tokens_total:,} tokens ({tokens_total/1e6:.1f}M) across 6 runs")
print(f"ckpt interval   : every {cfg.training.checkpoint_interval} steps "
      f"({cfg.training.checkpoint_interval * EFFECTIVE_BATCH_TOKENS / 1e6:.1f}M tokens)")

print("\n=== logging wiring checks ===")
print(f"  train loss        : log_interval={cfg.training.log_interval}  ✓")
print(f"  val loss (inline) : eval_interval={cfg.training.eval_interval}  "
      f"eval_max_batches={cfg.training.eval_max_batches}  ✓")
print("  per-layer |h_l|   : LayerInputMagnitudeTracker on every train step + final eval  ✓")
print("  per-layer |dL/dh| : same tracker (gradient hooks) logged with train metrics  ✓")
print("  layer-1 depth-attn: Layer1DepthAttentionProbe — Full: weight/mag for "
      "emb, l0_attn, l0_mlp; Block: emb, partial_after_l0  ✓")
print("  throughput        : tokens_per_sec, step_time_sec, elapsed_time_sec every log  ✓")
print(f"  benchmarks in-train: positionwise_steps={cfg.evaluation.positionwise_steps}  "
      f"(empty = off)  ✓")
print(f"  Drive artifacts   : checkpoints/ runs/ logs/ under {DRIVE_ROOT}")

print("\n=== 6-run plan ===")
for seed in SEEDS:
    print(f"seed={seed}")
    for plan in RUN_PLAN:
        print(f"  {plan['label']:14s} -> {run_names[(seed, plan['label'])]}")

print("\n=== A100 cost / wall-clock estimate (from measured 90M throughput) ===")
print(f"scale = token_scale×param_scale = {token_scale:.3f}×{param_scale:.3f} = {wall_scale:.3f}x")
print("90M measured hours/run (mean seeds 123/1337): "
      f"base={WALL_90M_HOURS['baseline']:.2f}h  full={WALL_90M_HOURS['full_attnres']:.2f}h  "
      f"block={WALL_90M_HOURS['block_attnres']:.2f}h")
for label, hours in est_hours.items():
    print(f"  {label:14s}: ~{hours:.1f} h / run  × 2 seeds = ~{2 * hours:.1f} h")
print(f"  TOTAL suite     : ~{suite_hours_est:.0f} h wall-clock (sequential)")
print(f"  @ ${A100_USD_PER_HOUR:.1f}/h A100 : ~${suite_hours_est * A100_USD_PER_HOUR:.0f}  "
      f"(ballpark; Colab Pro+ / on-demand rates vary ~$1.5–$3/h → "
      f"~${suite_hours_est * 1.5:.0f}–${suite_hours_est * 3.0:.0f})")
print("\nSTOP: review this report, run the VRAM probe, then set CONFIRM_LAUNCH=True.")


## VRAM probe: shared micro-batch for all variants

Identical hyperparameters across variants — one shared micro-batch. Probes **Full AttnRes** (worst case at L=12, wider d=896) and picks the largest micro-batch that fits while keeping effective batch = 65,536 tokens. YAML default is `8 × 8`; set `FORCE_MICRO_BATCH` to skip the probe.


In [ ]:
import gc

EFFECTIVE_BATCH_SEQUENCES = EFFECTIVE_BATCH_TOKENS // cfg.data.block_size  # 64
VRAM_HEADROOM = 0.85
FORCE_MICRO_BATCH = None  # e.g. 8 to skip probe and use YAML default
CANDIDATES = [8, 4, 2, 1]


def _peak_mib_for_microbatch(micro_batch: int) -> float:
    overrides = [
        f"experiment.seed={SEEDS[0]}",
        "model.architecture=attnres",
        "model.attnres.enabled=true",
        "model.attnres.mode=full",
        f"data.batch_size={micro_batch}",
    ]
    probe_cfg = load_config(CONFIG_PATH, overrides=overrides)
    model = build_model(probe_cfg.model).cuda()
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()
    input_ids = torch.randint(0, probe_cfg.model.vocab_size, (micro_batch, probe_cfg.data.block_size), device="cuda")
    targets = torch.randint(0, probe_cfg.model.vocab_size, (micro_batch, probe_cfg.data.block_size), device="cuda")
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        logits, _ = model(input_ids)
        loss = torch.nn.functional.cross_entropy(
            logits.reshape(-1, logits.size(-1)), targets.reshape(-1)
        )
    loss.backward()
    optimizer.step()
    peak = torch.cuda.max_memory_allocated() / (1024 ** 2)
    del model, optimizer, input_ids, targets, logits, loss
    gc.collect()
    torch.cuda.empty_cache()
    return peak


if FORCE_MICRO_BATCH is not None:
    if EFFECTIVE_BATCH_SEQUENCES % FORCE_MICRO_BATCH != 0:
        raise ValueError(f"FORCE_MICRO_BATCH={FORCE_MICRO_BATCH} must divide {EFFECTIVE_BATCH_SEQUENCES}")
    MICRO_BATCH = FORCE_MICRO_BATCH
    GRAD_ACCUM = EFFECTIVE_BATCH_SEQUENCES // FORCE_MICRO_BATCH
    print(f"FORCED micro_batch={MICRO_BATCH}, grad_accum={GRAD_ACCUM} "
          f"-> {MICRO_BATCH * GRAD_ACCUM * cfg.data.block_size:,} tokens/step (probe skipped)")
else:
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA required for VRAM probe")
    budget = VRAM_HEADROOM * (torch.cuda.get_device_properties(0).total_memory / (1024 ** 2))
    print(f"VRAM budget (headroom {VRAM_HEADROOM:.0%}): {budget:.0f} MiB")
    chosen = None
    for mb in CANDIDATES:
        if EFFECTIVE_BATCH_SEQUENCES % mb != 0:
            continue
        try:
            peak = _peak_mib_for_microbatch(mb)
            ok = peak <= budget
            print(f"  micro_batch={mb}: peak≈{peak:.0f} MiB  {'OK' if ok else 'OOM-risk'}")
            if ok:
                chosen = mb
                break
        except torch.cuda.OutOfMemoryError:
            print(f"  micro_batch={mb}: OOM")
            gc.collect()
            torch.cuda.empty_cache()
    if chosen is None:
        raise RuntimeError("No micro-batch fit Full AttnRes under VRAM headroom")
    MICRO_BATCH = chosen
    GRAD_ACCUM = EFFECTIVE_BATCH_SEQUENCES // MICRO_BATCH
    print(f"SELECTED micro_batch={MICRO_BATCH}, grad_accum={GRAD_ACCUM} "
          f"-> {MICRO_BATCH * GRAD_ACCUM * cfg.data.block_size:,} tokens/step (shared by all variants)")


## Confirmation gate (must be True to launch)

Set `CONFIRM_LAUNCH = True` only after reviewing the pre-launch report and VRAM probe. Leave `CLEAN_EXISTING_RUNS = False` unless you intentionally want to wipe Drive artifacts for these 6 run names before a fresh start.


In [ ]:
import shutil
from src.training.train import train_from_config

CONFIRM_LAUNCH = False  # <-- flip to True only after reviewing the report above
CLEAN_EXISTING_RUNS = False

SHARED_OVERRIDES = [
    f"logging.output_root={DRIVE_ROOT}",
    f"data.batch_size={MICRO_BATCH}",
    f"data.eval_batch_size={MICRO_BATCH}",
    f"training.grad_accum_steps={GRAD_ACCUM}",
    "logging.wandb.group=blessed_130m_ladder",
]

print("  CONFIRM_LAUNCH      =", CONFIRM_LAUNCH)
print("  CLEAN_EXISTING_RUNS =", CLEAN_EXISTING_RUNS)
print(f"  micro_batch/grad_accum = {MICRO_BATCH} / {GRAD_ACCUM}")
print(f"  est suite wall-clock   ≈ {suite_hours_est:.0f} h  "
      f"(~${suite_hours_est * A100_USD_PER_HOUR:.0f} @ ${A100_USD_PER_HOUR:.1f}/h)")
print("  block layout           =", actual_block_sizes)
print("\nPlanned runs:")
for seed in SEEDS:
    for plan in RUN_PLAN:
        print(f"  seed={seed}  {plan['label']:14s}  est≈{est_hours[plan['label']]:.1f}h  "
              f"run={run_names[(seed, plan['label'])]}")

if not CONFIRM_LAUNCH:
    raise RuntimeError(
        "Set CONFIRM_LAUNCH=True after reviewing the 130M report / block layout / "
        "cost estimate / VRAM probe, then rerun."
    )

results = []
suite_start = time.time()
for seed in SEEDS:
    for plan in RUN_PLAN:
        label = plan["label"]
        run_name = run_names[(seed, label)]
        print(f"\nLAUNCHING seed={seed} label={label} arch={plan['architecture']} "
              f"mb={MICRO_BATCH} ga={GRAD_ACCUM}")
        if CLEAN_EXISTING_RUNS:
            for sub in ("checkpoints", "runs", "logs"):
                path = DRIVE_ROOT / sub / run_name
                if path.exists():
                    print("removing", path)
                    shutil.rmtree(path)

        launch_cfg = load_config(
            CONFIG_PATH,
            overrides=[
                f"experiment.seed={seed}",
                *plan["overrides"],
                *SHARED_OVERRIDES,
                f"logging.wandb.run_name={run_name}",
            ],
        )
        run_start = time.time()
        summary = train_from_config(launch_cfg)
        run_hours = (time.time() - run_start) / 3600
        row = {
            "seed": seed,
            "label": label,
            "architecture": plan["architecture"],
            "run_name": run_name,
            "micro_batch": MICRO_BATCH,
            "grad_accum": GRAD_ACCUM,
            "wall_clock_hours": run_hours,
            "val_loss": summary.get("val_loss"),
            "best_val_loss": summary.get("best_val_loss"),
            "wandb_url": summary.get("wandb_url"),
            "mean_layer_input_magnitude_last_layer": summary.get(
                "mean_layer_input_magnitude_last_layer"
            ),
            "last_layer1_depth_attn": summary.get("last_layer1_depth_attn"),
        }
        results.append(row)
        print(f"Finished seed={seed} label={label} in {run_hours:.2f} h")
        print("  val_loss      :", summary.get("val_loss"))
        print("  best_val_loss :", summary.get("best_val_loss"))
        print("  last |h_L|    :", summary.get("mean_layer_input_magnitude_last_layer"))
        print("  L1 depth-attn :", summary.get("last_layer1_depth_attn"))
        print("  checkpoint    :", summary.get("checkpoint_path"))

suite_hours = (time.time() - suite_start) / 3600
_TRAINING_COMPLETED = True
print(f"\nALL RUNS COMPLETE in {suite_hours:.2f} h")
for row in results:
    print(
        f"  seed={row['seed']}  {row['label']:14s}  "
        f"val={row['val_loss']}  best={row['best_val_loss']}  wall={row['wall_clock_hours']:.2f}h"
    )


## End the Colab session on completion

Unassigns the runtime after all runs finish so idle compute is not billed.


In [ ]:
UNASSIGN_ON_COMPLETE = True

training_completed = globals().get("_TRAINING_COMPLETED", False)

if UNASSIGN_ON_COMPLETE and training_completed:
    from google.colab import runtime
    print("Unassigning runtime…")
    runtime.unassign()
elif not training_completed:
    print("Training not marked complete — leaving runtime assigned.")
else:
    print("UNASSIGN_ON_COMPLETE=False — leaving runtime assigned.")
